# Brief A (simple version): where is the business short of plan, and why?

**The question from the business:** "Nobody can say which part of the business is short."

This is the **same analysis** as `Data_Analysis_beginner.ipynb`, rewritten the way `session_5.ipynb` is written:

- one small idea per cell, and the result is shown right below the cell
- tables are named `df_...` and we look at them with `.head()`
- only the tools you already know: `read_csv`, `merge`, `groupby` / `agg`, `pivot_table`, `np.where`, simple column maths
- **no** classes, **no** helper functions, **no** `lambda`, **no** `argparse`

| Part | Block | Question |
|---|---|---|
| 0 | Setup | Load the files and clean them |
| 1 | A1 | City: prior vs current vs plan |
| 1 | A2 / A3 | Attainment by city and month, network totals |
| 1 | A4 | Channel view (with an assumption) |
| 1 | A5 | Revenue growth split: patients / tests per patient / revenue per test |
| 1 | A6 | Funnel: what happened to booked tests next? |
| 1 | A7 | Rupee value of the gap |
| 1 | A8 | Rank the cities by shortfall |
| 2 | R1 to R10 | Plan realism, levers, patient flow, peer gap, channel, clinics, test mix, system cutover, patient identity, lead time |
| 2 | R0 | Scorecard of computed facts |

**Part 1 finds *where* the shortfall is. Part 2 asks *why*. Part 2 narrows the causes, it cannot prove one.**

**Key definitions (used everywhere)**
- **Test volume** = number of rows in `booking_lines`. The plan (`plan_long`) is built on **booked** tests, not invoiced tests.
- **Month of a booking** = month of `slot_at` (service date) by default.
- **Superseded originals** = bookings that were rescheduled. We remove them from booked volume, because the replacement booking carries the same tests.
- **Revenue** = sum of `invoices.amount`, by invoice date.
- **Prior** = the same months one year earlier. **Current** = the months the plan covers.

**What is simpler than the original (scope cuts, on purpose)**

| Original | This notebook |
|---|---|
| `Report` class that prints and saves every table | Plain `df` variables. Add `df.to_csv("name.csv")` yourself if you want a file |
| A5 and R2: log-mean (Divisia) split into rupees / tests | Simple growth % of the three drivers (patients, tests per patient, revenue per test). They multiply to the total growth |
| R1: `np.polyfit` slope, run-rate ratio, shape correlation | Average attainment of the first 2 months vs last 2 months, plus one correlation across cities |
| Part 2 reloads and overwrites the data | Everything is loaded once in Part 0 and reused |

## Part 0. Setup
### Imports and settings

In [97]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [98]:
import numpy as np
import pandas as pd
from pathlib import Path

pd.options.display.float_format = "{:,.2f}".format
pd.options.display.max_columns = 50
pd.options.display.max_rows = 100
pd.options.display.width = 200

CRORE = 10_000_000     # 1 crore = 10,000,000 (we show rupees in crores)

### Settings (change them here, nowhere else)

- `DATA_DIR`: the folder with the `*.csv.gz` files
- `BASIS`: which date puts a booking into a month. `"slot_at"` = service date, `"created_at"` = booking date

In [99]:
# If your files are on Google Drive, un-comment the next two lines
# from google.colab import drive
# drive.mount('/content/drive')

DATA_DIR = Path("/content/drive/MyDrive/merged")      # <- change if your folder is different
BASIS = "slot_at"                     # "slot_at" or "created_at"

### Load the files

`pd.read_csv` reads `.csv.gz` directly, no need to unzip. We keep the phone number as text (`dtype`), so no digit is lost.

In [100]:
df_plan = pd.read_csv(DATA_DIR / "plan_long.csv.gz")
df_bookings = pd.read_csv(DATA_DIR / "bookings.csv.gz", low_memory=False, dtype={"patient_phone": str})
df_lines = pd.read_csv(DATA_DIR / "booking_lines.csv.gz")
df_collections = pd.read_csv(DATA_DIR / "collections.csv.gz")
df_invoices = pd.read_csv(DATA_DIR / "invoices.csv.gz")

/tmp/ipykernel_1348/3270172142.py:3: DtypeWarning: Columns (8,13,14,15) have mixed types. Specify dtype option on import or set low_memory=False.
  df_lines = pd.read_csv(DATA_DIR / "booking_lines.csv.gz")


In [101]:
# the campaigns file is optional: if it is missing we treat every booking as "no campaign"
try:
    df_campaigns = pd.read_csv(DATA_DIR / "campaigns.csv.gz")
except FileNotFoundError:
    df_campaigns = pd.DataFrame({"booking_id": []})
    print("No campaigns file found, so campaign = False for every booking")

In [102]:
display(df_plan.head())
display(df_bookings.head())
display(df_lines.head())
display(df_collections.head())
display(df_invoices.head())

,city,metric,month,target
0,Ahmedabad,test_volume_target_2026,2026-01,15119
1,Bengaluru,test_volume_target_2026,2026-01,20380
2,Chennai,test_volume_target_2026,2026-01,15474
3,Hyderabad,test_volume_target_2026,2026-01,14840
4,Jaipur,test_volume_target_2026,2026-01,8853


,booking_id,created_at,slot_at,city,clinic_id,channel,source,patient_id,patient_name,home_fee,source_system,rescheduled_from,status_at_export,patient.is_new,n_tests,tests_total,patient_phone,patient_dob,patient_signup_date,clinic_clinic_name,clinic_opened_on,clinic_has_lab,clinic_slots_per_day
0,BK-AHM-202501-000001,2025-01-08 03:16:00,2025-01-09 08:26:00,Ahmedabad,AHM-03,clinic,walkin,PAHM0028525,Saanvi Bose,0,legacy,NaN,NaN,NaN,3,"2,000.00",+91 76866 11670,1995-06-17,2024-01-11,Kalpa Health Ahmedabad 3,2023-06-01,0,40
1,BK-AHM-202501-000002,2025-01-03 05:18:00,2025-01-06 13:22:00,Ahmedabad,AHM-04,clinic,app,PAHM0031384,Nisha Iyer,0,legacy,NaN,NaN,NaN,1,650.00,+91 75793 12879,1997-03-17,2024-06-22,Kalpa Health Ahmedabad 4,2023-06-01,0,40
2,BK-AHM-202501-000003,2025-01-27 10:46:00,2025-01-28 13:37:00,Ahmedabad,AHM-01,home,app,PAHM0027001,Sneha Menon,150,legacy,NaN,NaN,NaN,4,"1,670.00",+91 94881 71237,1977-10-06,2023-10-23,Kalpa Health Ahmedabad 1,2023-06-01,1,60
3,BK-AHM-202501-000004,2025-01-20 02:48:00,2025-01-24 07:15:00,Ahmedabad,AHM-03,clinic,app,PAHM0007069,Arjun Bose,0,legacy,NaN,NaN,NaN,1,160.00,+91 96126 68447,1965-06-14,2024-11-12,Kalpa Health Ahmedabad 3,2023-06-01,0,40
4,BK-AHM-202501-000005,2025-01-06 08:21:00,2025-01-11 09:56:00,Ahmedabad,AHM-03,clinic,app,PAHM0002520,Ritika Nair,0,legacy,NaN,NaN,NaN,2,"1,900.00",+91 83270 41852,1986-10-07,2024-12-31,Kalpa Health Ahmedabad 3,2023-06-01,0,40


,booking_id,created_at,slot_at,city,clinic_id,channel,source,patient_id,patient_name,test_code,test_price,home_fee,source_system,rescheduled_from,status_at_export,patient.is_new,test_code_missing,test_name,list_price_rs,is_package,price_vs_list_rs,result_ts,result_value,result_flag
0,BK-AHM-202501-000001,2025-01-08 03:16:00,2025-01-09 08:26:00,Ahmedabad,AHM-03,clinic,walkin,PAHM0028525,Saanvi Bose,T009,800.00,0,legacy,NaN,NaN,NaN,False,Vitamin B12,800.00,0.00,0.00,2025-01-09 20:09:00,46.8,normal
1,BK-AHM-202501-000001,2025-01-08 03:16:00,2025-01-09 08:26:00,Ahmedabad,AHM-03,clinic,walkin,PAHM0028525,Saanvi Bose,T004,650.00,0,legacy,NaN,NaN,NaN,False,Lipid profile,650.00,0.00,0.00,2025-01-10 00:09:00,49.0,normal
2,BK-AHM-202501-000001,2025-01-08 03:16:00,2025-01-09 08:26:00,Ahmedabad,AHM-03,clinic,walkin,PAHM0028525,Saanvi Bose,T007,550.00,0,legacy,NaN,NaN,NaN,False,Thyroid profile,550.00,0.00,0.00,2025-01-09 18:09:00,74.8,normal
3,BK-AHM-202501-000002,2025-01-03 05:18:00,2025-01-06 13:22:00,Ahmedabad,AHM-04,clinic,app,PAHM0031384,Nisha Iyer,T004,650.00,0,legacy,NaN,NaN,NaN,False,Lipid profile,650.00,0.00,0.00,2025-01-07 15:44:00,52.7,flag
4,BK-AHM-202501-000003,2025-01-27 10:46:00,2025-01-28 13:37:00,Ahmedabad,AHM-01,home,app,PAHM0027001,Sneha Menon,T003,450.00,150,legacy,NaN,NaN,NaN,False,HbA1c,450.00,0.00,0.00,2025-01-29 05:37:00,71.7,flag


,booking_id,clinic_id,channel,status,collected_at,phlebotomist_id,patient_id,city,slot_at
0,BK-AHM-202501-000001,AHM-03,clinic,collected,2025-01-09 09:09:00,NaN,PAHM0028525,Ahmedabad,2025-01-09 08:26:00
1,BK-AHM-202501-000002,AHM-04,clinic,collected,2025-01-06 13:44:00,NaN,PAHM0031384,Ahmedabad,2025-01-06 13:22:00
2,BK-AHM-202501-000003,AHM-01,home,collected,2025-01-28 13:37:00,PHAHM047,PAHM0027001,Ahmedabad,2025-01-28 13:37:00
3,BK-AHM-202501-000005,AHM-03,clinic,collected,2025-01-11 09:59:00,NaN,PAHM0002520,Ahmedabad,2025-01-11 09:56:00
4,BK-AHM-202501-000006,AHM-04,home,collected,2025-01-11 12:39:00,PHAHM011,PAHM0018893,Ahmedabad,2025-01-11 11:28:00


,invoice_id,invoice_date,booking_id,patient_id,city,channel,amount,source_file
0,INV-2024-000001,2024-12-30,BK-BLR-202501-006763,PBLR0044005,Bengaluru,clinic,280.00,datapack/invoices/invoices_2024Q4.csv.gz
1,INV-2024-000002,2024-12-30,BK-BLR-202501-007103,PBLR0044345,Bengaluru,clinic,"1,070.00",datapack/invoices/invoices_2024Q4.csv.gz
2,INV-2024-000003,2024-12-31,BK-PUN-202501-001985,PPUN0008860,Pune,clinic,350.00,datapack/invoices/invoices_2024Q4.csv.gz
3,INV-2024-000004,2024-12-31,BK-CHE-202501-001176,PCHE0012203,Chennai,home,"3,299.00",datapack/invoices/invoices_2024Q4.csv.gz
4,INV-2025-000001,2025-01-01,BK-AHM-202501-005119,PAHM0034226,Ahmedabad,clinic,"2,649.00",datapack/invoices/invoices_2025Q1.csv.gz


In [103]:
df_bookings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 568007 entries, 0 to 568006
Data columns (total 23 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   booking_id            568007 non-null  object 
 1   created_at            568007 non-null  object 
 2   slot_at               568007 non-null  object 
 3   city                  568007 non-null  object 
 4   clinic_id             568007 non-null  object 
 5   channel               568007 non-null  object 
 6   source                568007 non-null  object 
 7   patient_id            568007 non-null  object 
 8   patient_name          541464 non-null  object 
 9   home_fee              568007 non-null  int64  
 10  source_system         568007 non-null  object 
 11  rescheduled_from      1500 non-null    object 
 12  status_at_export      26543 non-null   object 
 13  patient.is_new        26543 non-null   object 
 14  n_tests               568007 non-null  int64  
 15  

### Clean the plan table

Remove hidden spaces from `month` and `city`. A hidden space is the classic reason a join silently returns nothing.

In [104]:
df_plan["month"] = df_plan["month"].astype(str).str.strip()
df_plan["city"] = df_plan["city"].astype(str).str.strip()

df_plan.head()

,city,metric,month,target
0,Ahmedabad,test_volume_target_2026,2026-01,15119
1,Bengaluru,test_volume_target_2026,2026-01,20380
2,Chennai,test_volume_target_2026,2026-01,15474
3,Hyderabad,test_volume_target_2026,2026-01,14840
4,Jaipur,test_volume_target_2026,2026-01,8853


### Clean the bookings table

Steps:
1. remember the raw row count, so we can report how many duplicates we drop
2. clean `booking_id`, drop duplicate bookings
3. clean `city` and `rescheduled_from`
4. turn the date columns into real dates (a bad value becomes blank, `NaT`)

In [105]:
rows_raw = df_bookings.shape[0]

df_bookings["booking_id"] = df_bookings["booking_id"].astype(str).str.strip()
df_bookings = df_bookings.drop_duplicates("booking_id").copy()

df_bookings["city"] = df_bookings["city"].astype(str).str.strip()
df_bookings["rescheduled_from"] = df_bookings["rescheduled_from"].astype("object").str.strip()

df_bookings["created_at"] = pd.to_datetime(df_bookings["created_at"], errors="coerce")
df_bookings["slot_at"] = pd.to_datetime(df_bookings["slot_at"], errors="coerce")
df_bookings["patient_signup_date"] = pd.to_datetime(df_bookings["patient_signup_date"], errors="coerce")

print(f"{rows_raw:,} raw rows, {rows_raw - df_bookings.shape[0]:,} duplicate booking_id removed")

568,007 raw rows, 0 duplicate booking_id removed


### Superseded bookings (reschedules)

When a booking is rescheduled, the **new** booking stores the **old** id in `rescheduled_from`. So every id that appears in `rescheduled_from` is an old booking that was replaced. We flag those, and remove them from booked volume later.

In [106]:
superseded_ids = df_bookings["rescheduled_from"].dropna().tolist()
df_bookings["superseded"] = df_bookings["booking_id"].isin(superseded_ids)

df_bookings["superseded"].value_counts()

,count
superseded,
False,566507
True,1500


### Tests per booking, and the month of each booking

- `df_lines.groupby("booking_id").size()` counts the test lines of each booking
- `merge` copies that count onto the bookings. A booking with no lines gets 0
- `month` puts the booking into a month, using the date chosen in `BASIS`

In [107]:
df_lines["booking_id"] = df_lines["booking_id"].astype(str).str.strip()

tests_per_booking = df_lines.groupby("booking_id").size().reset_index(name="tests")
tests_per_booking.head()

,booking_id,tests
0,BK-AHM-202501-000001,3
1,BK-AHM-202501-000002,1
2,BK-AHM-202501-000003,4
3,BK-AHM-202501-000004,1
4,BK-AHM-202501-000005,2


In [108]:
df_bookings = df_bookings.merge(tests_per_booking, on="booking_id", how="left")
df_bookings["tests"] = df_bookings["tests"].fillna(0).astype(int)

df_bookings["month"] = df_bookings[BASIS].dt.to_period("M").astype(str)

df_bookings[["booking_id", "city", BASIS, "month", "tests"]].head()

,booking_id,city,slot_at,month,tests
0,BK-AHM-202501-000001,Ahmedabad,2025-01-09 08:26:00,2025-01,3
1,BK-AHM-202501-000002,Ahmedabad,2025-01-06 13:22:00,2025-01,1
2,BK-AHM-202501-000003,Ahmedabad,2025-01-28 13:37:00,2025-01,4
3,BK-AHM-202501-000004,Ahmedabad,2025-01-24 07:15:00,2025-01,1
4,BK-AHM-202501-000005,Ahmedabad,2025-01-11 09:56:00,2025-01,2


### Collection status

Every booking becomes `collected`, `no_show` or **`unknown`** (no record at all). `unknown` is **not** the same as collected or no-show, so it gets its own label.

In [109]:
df_collections["booking_id"] = df_collections["booking_id"].astype(str).str.strip()
df_collections = df_collections.drop_duplicates("booking_id")

df_status = df_collections[["booking_id", "status"]].rename(columns={"status": "coll_status"})
df_bookings = df_bookings.merge(df_status, on="booking_id", how="left")
df_bookings["coll_status"] = df_bookings["coll_status"].fillna("unknown")

df_bookings["coll_status"].value_counts()

,count
coll_status,
collected,470426
no_show,79172
unknown,18409


### Clean the invoices

Same pattern: clean the text columns, convert date and amount (`errors="coerce"` turns bad values into blanks), drop the rows with a bad date or amount, then add `month` and `samples` (number of tests on the invoiced booking).

In [110]:
df_invoices["booking_id"] = df_invoices["booking_id"].astype(str).str.strip()
df_invoices["city"] = df_invoices["city"].astype(str).str.strip()
df_invoices["invoice_date"] = pd.to_datetime(df_invoices["invoice_date"], errors="coerce")
df_invoices["amount"] = pd.to_numeric(df_invoices["amount"], errors="coerce")

invoices_raw = df_invoices.shape[0]
df_invoices = df_invoices.dropna(subset=["invoice_date", "amount"]).copy()
print(f"{invoices_raw - df_invoices.shape[0]:,} invoices dropped for a bad date or amount")

df_invoices["month"] = df_invoices["invoice_date"].dt.to_period("M").astype(str)

0 invoices dropped for a bad date or amount


In [111]:
df_samples = tests_per_booking.rename(columns={"tests": "samples"})
df_invoices = df_invoices.merge(df_samples, on="booking_id", how="left")
df_invoices["samples"] = df_invoices["samples"].fillna(0).astype(int)

# a booking is "invoiced" if its booking_id appears in the invoices table
df_bookings["invoiced"] = df_bookings["booking_id"].isin(df_invoices["booking_id"])

df_bookings["invoiced"].value_counts()

,count
invoiced,
True,470776
False,97231


### Reschedule pairs share one outcome (needed only for the funnel A6)

The visit and the invoice sit on the **original** booking. The replacement is never collected or invoiced. For the funnel we copy the original's status and invoice flag onto the replacement, so a pair counts **once**.

- build a small table of originals (`df_original`) with the column renamed to `rescheduled_from`
- `merge` it onto the bookings: every replacement now sees its original's status
- `np.where(is_replacement, original value, own value)` picks the right one per row

In [112]:
df_original = df_bookings[["booking_id", "coll_status", "invoiced"]].rename(
    columns={"booking_id": "rescheduled_from", "coll_status": "orig_status", "invoiced": "orig_invoiced"})
df_bookings = df_bookings.merge(df_original, on="rescheduled_from", how="left")

is_replacement = df_bookings["rescheduled_from"].notna()
df_bookings["orig_invoiced"] = df_bookings["orig_invoiced"].eq(True)

df_bookings["eff_status"] = np.where(is_replacement, df_bookings["orig_status"].fillna("unknown"), df_bookings["coll_status"])
df_bookings["eff_invoiced"] = np.where(is_replacement, df_bookings["orig_invoiced"], df_bookings["invoiced"])

print(f"{is_replacement.sum():,} replacement bookings take their original's outcome")

1,500 replacement bookings take their original's outcome


### Data checks

Each line is one fact. A check passes when the number is 0.

In [113]:
lines_without_booking = (~df_lines["booking_id"].isin(df_bookings["booking_id"])).sum()
bookings_without_lines = (df_bookings["tests"] == 0).sum()
invoices_without_booking = (~df_invoices["booking_id"].isin(df_bookings["booking_id"])).sum()
missing_test_code = df_lines["test_code"].isna().sum()
no_collection = (df_bookings.loc[~df_bookings["superseded"], "coll_status"] == "unknown").mean()

print(f"Lines without a booking:          {lines_without_booking:,}   (0 is good)")
print(f"Bookings with zero test lines:    {bookings_without_lines:,}   (0 is good)")
print(f"Invoices without a booking:       {invoices_without_booking:,}   (0 is good)")
print(f"Lines with a missing test_code:   {missing_test_code:,}   (kept in the volume)")
print(f"Bookings with no collection record: {no_collection:.1%}   (labelled 'unknown')")
print("Source systems:", df_bookings["source_system"].value_counts().to_dict())
print("Plan metrics:  ", df_plan["metric"].unique().tolist(), "(the plan is on a BOOKED-tests basis)")

Lines without a booking:          0   (0 is good)
Bookings with zero test lines:    0   (0 is good)
Invoices without a booking:       0   (0 is good)
Lines with a missing test_code:   1,311   (kept in the volume)
Bookings with no collection record: 3.2%   (labelled 'unknown')
Source systems: {'legacy': 541464, 'nova': 26543}
Plan metrics:   ['test_volume_target_2026'] (the plan is on a BOOKED-tests basis)


## 1.1 Analysis setup

- `cur_months` = the months the plan covers
- `prior_months` = the same months one year earlier (a `PeriodIndex` minus 12 months)
- `period` column: `current`, `prior` or `other`
- `compare_month` column: for a prior row it is the **current** month it lines up with (Jan 2025 -> 2026-01). Then prior and current can sit in the same column of a table

In [114]:
cur_months = sorted(df_plan["month"].unique())
prior_months = (pd.PeriodIndex(cur_months, freq="M") - 12).astype(str)

print("Current months:", cur_months)
print("Prior months:  ", list(prior_months))

Current months: ['2026-01', '2026-02', '2026-03', '2026-04', '2026-05', '2026-06']
Prior months:   ['2025-01', '2025-02', '2025-03', '2025-04', '2025-05', '2025-06']


In [115]:
prior_to_current = dict(zip(prior_months, cur_months))
prior_to_current

{'2025-01': '2026-01',
 '2025-02': '2026-02',
 '2025-03': '2026-03',
 '2025-04': '2026-04',
 '2025-05': '2026-05',
 '2025-06': '2026-06'}

In [116]:
df_bookings["period"] = np.where(df_bookings["month"].isin(cur_months), "current",
                        np.where(df_bookings["month"].isin(prior_months), "prior", "other"))
df_bookings["compare_month"] = df_bookings["month"].replace(prior_to_current)

df_bookings[["booking_id", "month", "period", "compare_month"]].sample(5)

,booking_id,month,period,compare_month
547210,nova_che_0093302,2026-04,current,2026-04
382610,BK-HYD-202601-063292,2026-01,current,2026-01
532132,BK-PUN-202604-070648,2026-04,current,2026-04
181731,BK-BLR-202512-080149,2025-12,other,2025-12
314078,BK-CHE-202603-082315,2026-03,current,2026-03


Two more columns we need in Part 2:
- `lead_days` = days between booking and service date
- `campaign` = True if the booking is in the campaigns file

In [117]:
df_bookings["lead_days"] = (df_bookings["slot_at"] - df_bookings["created_at"]).dt.total_seconds() / 86400

campaign_ids = df_campaigns["booking_id"].astype(str).str.strip().tolist()
df_bookings["campaign"] = df_bookings["booking_id"].isin(campaign_ids)

df_bookings[["lead_days", "campaign"]].describe(include="all")

,lead_days,campaign
count,"568,007.00",568007
unique,NaN,2
top,NaN,False
freq,NaN,519138
mean,2.71,NaN
std,1.73,NaN
min,-2.00,NaN
25%,1.20,NaN
50%,3.00,NaN
75%,4.21,NaN


Now the two tables we use for almost everything:
- `df_booked` = bookings **without** the superseded originals (the "booked" basis of the plan)
- `df_compare` = `df_booked`, only the rows in the prior or current period

In [118]:
df_booked = df_bookings[~df_bookings["superseded"]].copy()
df_compare = df_booked[df_booked["period"] != "other"].copy()

print(df_bookings.shape[0], df_booked.shape[0], df_compare.shape[0])

568007 566507 384921


The same two columns for the invoices, so we can compare prior and current revenue.

In [119]:
df_invoices["period"] = np.where(df_invoices["month"].isin(cur_months), "current",
                        np.where(df_invoices["month"].isin(prior_months), "prior", "other"))
df_invoices["compare_month"] = df_invoices["month"].replace(prior_to_current)

df_inv_compare = df_invoices[df_invoices["period"] != "other"].copy()
df_inv_compare["period"].value_counts()

,count
period,
current,162876
prior,157451


### Check before we compare plan and actual

1. Do the plan cities and the booking cities match exactly? (`^` shows the cities that are in only one of the two)
2. Is the last plan month **complete** in the bookings? If the bookings stop mid-month, that month looks short for no real reason

In [120]:
only_in_one = set(df_plan["city"]) ^ set(df_booked["city"])
print("Cities in only one source:", only_in_one if only_in_one else "none, they match")

last_plan_month_end = pd.to_datetime(cur_months[-1] + "-01") + pd.offsets.MonthEnd(0)
latest_booking = df_booked[BASIS].max()
print(f"Latest {BASIS}: {latest_booking:%Y-%m-%d}, last plan month ends: {last_plan_month_end:%Y-%m-%d}")
print("Last plan month complete?", latest_booking >= last_plan_month_end.normalize())

Cities in only one source: none, they match
Latest slot_at: 2026-07-04, last plan month ends: 2026-06-30
Last plan month complete? True


# PART 1. Locate the shortfall

## 1.2 A1: Plan vs actual by city

For each city:
- **prior_booked**, **current_booked**, **current_plan** (tests)
- actual growth, the growth the plan asked for, and the gap between them in percentage points (pp)
- **attainment** = booked / plan
- **gap_tests** = plan - booked

We add the TOTAL row **before** the ratios, so total growth is a ratio of totals, not an average of city percentages.

New ideas: `pivot_table` (cities down, periods across), `.join`, `.loc["TOTAL"] = .sum()`.

In [121]:
city_table = df_compare.pivot_table(index="city", columns="period", values="tests", aggfunc="sum", fill_value=0)
city_table = city_table.rename(columns={"prior": "prior_booked", "current": "current_booked"})
city_table

period,current_booked,prior_booked
city,,
Ahmedabad,85084,76926
Bengaluru,107661,100070
Chennai,80772,80267
Hyderabad,78674,73237
Jaipur,40236,45149
Pune,65575,60653


In [122]:
plan_by_city = df_plan[df_plan["month"].isin(cur_months)].groupby("city").agg(current_plan=("target", "sum"))

# join = merge on the index (here the index is the city)
city_table = city_table.join(plan_by_city)
city_table = city_table[["prior_booked", "current_booked", "current_plan"]]
city_table.loc["TOTAL"] = city_table.sum()
city_table

,prior_booked,current_booked,current_plan
city,,,
Ahmedabad,76926,85084,90773
Bengaluru,100070,107661,120085
Chennai,80267,80772,93912
Hyderabad,73237,78674,86421
Jaipur,45149,40236,52372
Pune,60653,65575,72177
TOTAL,436302,458002,515740


In [123]:
city_table["actual_growth_pct"] = (city_table["current_booked"] / city_table["prior_booked"] - 1) * 100
city_table["plan_growth_pct"] = (city_table["current_plan"] / city_table["prior_booked"] - 1) * 100
city_table["growth_gap_pp"] = city_table["actual_growth_pct"] - city_table["plan_growth_pct"]
city_table["attainment_pct"] = city_table["current_booked"] / city_table["current_plan"] * 100
city_table["gap_tests"] = city_table["current_plan"] - city_table["current_booked"]

city_table.round(1)

,prior_booked,current_booked,current_plan,actual_growth_pct,plan_growth_pct,growth_gap_pp,attainment_pct,gap_tests
city,,,,,,,,
Ahmedabad,76926,85084,90773,10.60,18.00,-7.40,93.70,5689
Bengaluru,100070,107661,120085,7.60,20.00,-12.40,89.70,12424
Chennai,80267,80772,93912,0.60,17.00,-16.40,86.00,13140
Hyderabad,73237,78674,86421,7.40,18.00,-10.60,91.00,7747
Jaipur,45149,40236,52372,-10.90,16.00,-26.90,76.80,12136
Pune,60653,65575,72177,8.10,19.00,-10.90,90.90,6602
TOTAL,436302,458002,515740,5.00,18.20,-13.20,88.80,57738


## 1.3 A2 and A3: Attainment by month

- `actual_cm` = booked tests, city x month
- `plan_cm` = plan, city x month
- `attainment_cm` = actual / plan, in %
- `monthly_totals` = network totals per month

Look for: a city weak in **every** month (a structural problem), or one that fades over time (losing momentum).

In [124]:
df_cur = df_booked[df_booked["period"] == "current"]

actual_cm = df_cur.pivot_table(index="city", columns="month", values="tests", aggfunc="sum", fill_value=0)
plan_cm = df_plan[df_plan["month"].isin(cur_months)].pivot_table(index="city", columns="month", values="target", aggfunc="sum")

attainment_cm = (actual_cm / plan_cm * 100).round(1)
attainment_cm

month,2026-01,2026-02,2026-03,2026-04,2026-05,2026-06
city,,,,,,
Ahmedabad,92.60,98.20,97.00,89.30,93.00,92.60
Bengaluru,89.10,98.60,83.00,92.00,89.50,86.10
Chennai,88.50,97.70,80.70,82.80,84.80,82.30
Hyderabad,88.90,91.90,93.30,88.50,88.40,95.50
Jaipur,76.40,72.60,72.00,80.30,76.80,84.60
Pune,93.20,87.80,97.80,90.70,84.10,91.60


In [125]:
print("City-month cells missing in plan or bookings:", attainment_cm.isna().sum().sum())

monthly_totals = pd.DataFrame({"plan": plan_cm.sum(), "actual": actual_cm.sum()})
monthly_totals["attainment_pct"] = monthly_totals["actual"] / monthly_totals["plan"] * 100
monthly_totals["gap"] = monthly_totals["plan"] - monthly_totals["actual"]
monthly_totals.round(1)

City-month cells missing in plan or bookings: 0


,plan,actual,attainment_pct,gap
month,,,,
2026-01,87002,77321,88.90,9681
2026-02,86293,80123,92.80,6170
2026-03,87882,76770,87.40,11112
2026-04,85616,75287,87.90,10329
2026-05,84678,73717,87.10,10961
2026-06,84269,74784,88.70,9485


## 1.4 A4: Channel gap (city x channel)

The plan has **no channel split**, so we make an **assumption**: each city's plan growth applies equally to every channel in that city ("pro-rata").

- pro-rata plan = prior x (1 + city plan growth)
- gap = pro-rata plan - current
- share of city gap = this channel's gap / the city's total gap

Treat this as an indication only, because the assumption may not be true.

New idea: `groupby("city")[...].transform("sum")` = the city's total, repeated on every row of that city.

In [126]:
channel_table = df_compare.pivot_table(index=["city", "channel"], columns="period", values="tests", aggfunc="sum", fill_value=0)
channel_table = channel_table.reset_index()
channel_table.columns.name = None

channel_table.head()

,city,channel,current,prior
0,Ahmedabad,clinic,47380,50357
1,Ahmedabad,home,37704,26569
2,Bengaluru,clinic,59356,66025
3,Bengaluru,home,48305,34045
4,Chennai,clinic,44697,53032


In [127]:
channel_table["city_plan_growth_pct"] = channel_table["city"].map(city_table["plan_growth_pct"])
channel_table["actual_growth_pct"] = (channel_table["current"] / channel_table["prior"] - 1) * 100
channel_table["pro_rata_plan"] = channel_table["prior"] * (1 + channel_table["city_plan_growth_pct"] / 100)
channel_table["gap_vs_pro_rata_plan"] = channel_table["pro_rata_plan"] - channel_table["current"]

channel_table["city_gap"] = channel_table.groupby("city")["gap_vs_pro_rata_plan"].transform("sum")
channel_table["share_of_city_gap_pct"] = channel_table["gap_vs_pro_rata_plan"] / channel_table["city_gap"] * 100

channel_table.round(1)

,city,channel,current,prior,city_plan_growth_pct,actual_growth_pct,pro_rata_plan,gap_vs_pro_rata_plan,city_gap,share_of_city_gap_pct
0,Ahmedabad,clinic,47380,50357,18.00,-5.90,"59,421.50","12,041.50","5,689.00",211.70
1,Ahmedabad,home,37704,26569,18.00,41.90,"31,351.50","-6,352.50","5,689.00",-111.70
2,Bengaluru,clinic,59356,66025,20.00,-10.10,"79,230.70","19,874.70","12,424.00",160.00
3,Bengaluru,home,48305,34045,20.00,41.90,"40,854.30","-7,450.70","12,424.00",-60.00
4,Chennai,clinic,44697,53032,17.00,-15.70,"62,047.20","17,350.20","13,140.00",132.00
5,Chennai,home,36075,27235,17.00,32.50,"31,864.80","-4,210.20","13,140.00",-32.00
6,Hyderabad,clinic,43386,48187,18.00,-10.00,"56,861.50","13,475.50","7,747.00",173.90
7,Hyderabad,home,35288,25050,18.00,40.90,"29,559.50","-5,728.50","7,747.00",-73.90
8,Jaipur,clinic,22338,29685,16.00,-24.70,"34,434.00","12,096.00","12,136.00",99.70
9,Jaipur,home,17898,15464,16.00,15.70,"17,938.00",40.00,"12,136.00",0.30


## 1.5 A5: Why did revenue grow?

**Revenue = patients x (tests per patient) x (revenue per test)**

So revenue can grow because of more patients, patients buying more tests, or a higher price per test. Because the three multiply:

(1 + revenue growth) = (1 + patient growth) x (1 + tests-per-patient growth) x (1 + revenue-per-test growth)

We compute the growth of each driver and look at which one is far from the others.

**Trick for the TOTAL row:** a patient who visited two cities must count **once** in the total. So we stack a copy of the table with `city = "TOTAL"` under the original, and run the same `pivot_table` on it. `nunique` then counts distinct patients correctly.

In [128]:
df_inv_total = df_inv_compare.copy()
df_inv_total["city"] = "TOTAL"
df_inv_all = pd.concat([df_inv_compare, df_inv_total])

df_inv_all["city"].value_counts()

,count
city,
TOTAL,320327
Bengaluru,74522
Ahmedabad,58010
Chennai,57413
Hyderabad,54369
Pune,45319
Jaipur,30694


In [129]:
revenue = df_inv_all.pivot_table(index="city", columns="period", values="amount", aggfunc="sum")
tests = df_inv_all.pivot_table(index="city", columns="period", values="samples", aggfunc="sum")
patients = df_inv_all.pivot_table(index="city", columns="period", values="patient_id", aggfunc="nunique")

revenue

period,current,prior
city,,
Ahmedabad,"49,962,547.00","42,642,432.00"
Bengaluru,"63,055,882.00","54,957,270.00"
Chennai,"47,137,163.00","43,898,409.00"
Hyderabad,"45,983,558.00","39,907,314.00"
Jaipur,"23,811,659.00","24,800,605.00"
Pune,"38,332,364.00","33,376,677.00"
TOTAL,"268,283,173.00","239,582,707.00"


In [130]:
driver_table = pd.DataFrame({
    "revenue_prior_cr": revenue["prior"] / CRORE,
    "revenue_current_cr": revenue["current"] / CRORE,
    "revenue_growth_pct": (revenue["current"] / revenue["prior"] - 1) * 100,
    "patients_growth_pct": (patients["current"] / patients["prior"] - 1) * 100,
    "tests_per_patient_growth_pct": ((tests["current"] / patients["current"]) / (tests["prior"] / patients["prior"]) - 1) * 100,
    "revenue_per_test_growth_pct": ((revenue["current"] / tests["current"]) / (revenue["prior"] / tests["prior"]) - 1) * 100,
})

driver_table.round(2)

,revenue_prior_cr,revenue_current_cr,revenue_growth_pct,patients_growth_pct,tests_per_patient_growth_pct,revenue_per_test_growth_pct
city,,,,,,
Ahmedabad,4.26,5.00,17.17,10.23,0.60,5.66
Bengaluru,5.50,6.31,14.74,7.96,0.91,5.31
Chennai,4.39,4.71,7.38,-4.25,5.70,6.10
Hyderabad,3.99,4.60,15.23,8.42,-0.50,6.82
Jaipur,2.48,2.38,-3.99,-11.40,1.59,6.67
Pune,3.34,3.83,14.85,8.32,0.34,5.67
TOTAL,23.96,26.83,11.98,4.24,1.40,5.95


**How to read it:** the driver with the biggest growth is the main reason revenue grew (or the smallest one is the weak spot). Example: if patients grow +3%, tests per patient +10% and revenue per test +1%, growth came mostly from patients buying more tests, not from more patients.

## 1.6 A6: Funnel, what happened to the booked tests?

Each booked test ends in exactly one outcome:

| Outcome | Meaning |
|---|---|
| no_show | the patient did not come |
| unknown_not_invoiced | no collection record and no invoice (possibly lost) |
| unknown_invoiced | no collection record but billed (a missing record, not a missing visit) |
| collected_not_invoiced | sample taken but never billed (a revenue leak) |
| collected_invoiced | the healthy case |

We show each city's **share** (%) per outcome and the **change in percentage points** vs last year. A rising `no_show` or `collected_not_invoiced` share is a leak.

In [131]:
df_compare["invoice_label"] = np.where(df_compare["eff_invoiced"], "invoiced", "not_invoiced")
df_compare["outcome"] = np.where(df_compare["eff_status"] == "no_show", "no_show",
                                 df_compare["eff_status"] + "_" + df_compare["invoice_label"])

df_compare["outcome"].value_counts()

,count
outcome,
collected_invoiced,320044
no_show,53369
unknown_not_invoiced,11258
unknown_invoiced,250


In [132]:
funnel_prior = df_compare[df_compare["period"] == "prior"].pivot_table(
    index="city", columns="outcome", values="tests", aggfunc="sum", fill_value=0)
funnel_current = df_compare[df_compare["period"] == "current"].pivot_table(
    index="city", columns="outcome", values="tests", aggfunc="sum", fill_value=0)

funnel_prior.loc["TOTAL"] = funnel_prior.sum()
funnel_current.loc["TOTAL"] = funnel_current.sum()
funnel_current

outcome,collected_invoiced,no_show,unknown_invoiced,unknown_not_invoiced
city,,,,
Ahmedabad,70777,11738,54,2515
Bengaluru,90041,14412,83,3125
Chennai,67084,11331,33,2324
Hyderabad,65647,10517,58,2452
Jaipur,33708,5350,20,1158
Pune,54730,8877,29,1939
TOTAL,381987,62225,277,13513


In [133]:
# share of each outcome inside its city (each row adds up to 100)
share_prior = funnel_prior.div(funnel_prior.sum(axis=1), axis=0) * 100
share_current = funnel_current.div(funnel_current.sum(axis=1), axis=0) * 100

share_change_pp = share_current.subtract(share_prior, fill_value=0)

display(share_current.round(1))
display(share_change_pp.round(1))

outcome,collected_invoiced,no_show,unknown_invoiced,unknown_not_invoiced
city,,,,
Ahmedabad,83.20,13.80,0.10,3.00
Bengaluru,83.60,13.40,0.10,2.90
Chennai,83.10,14.00,0.00,2.90
Hyderabad,83.40,13.40,0.10,3.10
Jaipur,83.80,13.30,0.00,2.90
Pune,83.50,13.50,0.00,3.00
TOTAL,83.40,13.60,0.10,3.00


outcome,collected_invoiced,no_show,unknown_invoiced,unknown_not_invoiced
city,,,,
Ahmedabad,0.20,-0.30,0.00,0.10
Bengaluru,1.00,-0.90,0.00,-0.10
Chennai,0.50,-0.50,-0.10,0.10
Hyderabad,0.40,-0.50,0.00,0.20
Jaipur,0.70,-0.60,0.00,-0.20
Pune,0.50,-0.60,-0.00,0.20
TOTAL,0.60,-0.60,-0.00,0.00


## 1.7 A7: What is the volume gap worth in rupees?

value of gap = gap (tests) x conversion (booked -> invoiced) x revenue per invoiced test, in crores.

Conversion and price are measured **per city** in the current period. This is **indicative**: it assumes the missing tests would have converted and earned like an average test.

In [134]:
inv_current = df_inv_compare[df_inv_compare["period"] == "current"].groupby("city").agg(
    invoiced_tests=("samples", "sum"), revenue=("amount", "sum"))

value_table = city_table.drop("TOTAL")[["gap_tests", "current_booked"]].join(inv_current)
value_table.head()

,gap_tests,current_booked,invoiced_tests,revenue
city,,,,
Ahmedabad,5689,85084,70828,"49,962,547.00"
Bengaluru,12424,107661,90123,"63,055,882.00"
Chennai,13140,80772,67160,"47,137,163.00"
Hyderabad,7747,78674,65702,"45,983,558.00"
Jaipur,12136,40236,33770,"23,811,659.00"


In [135]:
value_table["conversion"] = value_table["invoiced_tests"] / value_table["current_booked"]
value_table["revenue_per_invoiced_test"] = value_table["revenue"] / value_table["invoiced_tests"]
value_table["value_of_gap_cr"] = (value_table["gap_tests"] * value_table["conversion"]
                                  * value_table["revenue_per_invoiced_test"] / CRORE)

display(value_table.round(3))
print(f"Total gap: {value_table['gap_tests'].sum():,.0f} tests, worth about {value_table['value_of_gap_cr'].sum():,.2f} crore (indicative)")

,gap_tests,current_booked,invoiced_tests,revenue,conversion,revenue_per_invoiced_test,value_of_gap_cr
city,,,,,,,
Ahmedabad,5689,85084,70828,"49,962,547.00",0.83,705.41,0.33
Bengaluru,12424,107661,90123,"63,055,882.00",0.84,699.66,0.73
Chennai,13140,80772,67160,"47,137,163.00",0.83,701.86,0.77
Hyderabad,7747,78674,65702,"45,983,558.00",0.83,699.88,0.45
Jaipur,12136,40236,33770,"23,811,659.00",0.84,705.11,0.72
Pune,6602,65575,54754,"38,332,364.00",0.83,700.08,0.39


Total gap: 57,738 tests, worth about 3.39 crore (indicative)


## 1.8 A8: Rank the cities by shortfall

For each city: the gap, the growth gap, the attainment, its **share of the total gap**, its **worst month** and how many months were below 90%.

Two ranks: by test gap and by growth gap. A big city can top the "tests" rank while a small city tops the "growth" rank, and both matter.

New idea: `idxmin(axis=1)` gives the **column name** of the smallest value in each row (here, the worst month).

In [136]:
ranking = city_table.drop("TOTAL")[["gap_tests", "growth_gap_pp", "attainment_pct", "actual_growth_pct"]].copy()

ranking["share_of_total_gap_pct"] = ranking["gap_tests"] / ranking["gap_tests"].sum() * 100
ranking["worst_month"] = attainment_cm.idxmin(axis=1)
ranking["worst_month_attainment_pct"] = attainment_cm.min(axis=1)
ranking["months_below_90"] = (attainment_cm < 90).sum(axis=1)

ranking["rank_by_gap"] = ranking["gap_tests"].rank(ascending=False).astype(int)
ranking["rank_by_growth_gap"] = ranking["growth_gap_pp"].rank(ascending=True).astype(int)

ranking.sort_values("gap_tests", ascending=False).round(1)

,gap_tests,growth_gap_pp,attainment_pct,actual_growth_pct,share_of_total_gap_pct,worst_month,worst_month_attainment_pct,months_below_90,rank_by_gap,rank_by_growth_gap
city,,,,,,,,,,
Chennai,13140,-16.40,86.00,0.60,22.80,2026-03,80.70,5,1,2
Bengaluru,12424,-12.40,89.70,7.60,21.50,2026-03,83.00,4,2,3
Jaipur,12136,-26.90,76.80,-10.90,21.00,2026-03,72.00,6,3,1
Hyderabad,7747,-10.60,91.00,7.40,13.40,2026-05,88.40,3,4,5
Pune,6602,-10.90,90.90,8.10,11.40,2026-05,84.10,2,5,4
Ahmedabad,5689,-7.40,93.70,10.60,9.90,2026-04,89.30,1,6,6


**Part 1 answer:** the table above says **where** the plan is missed (which cities, which months, which channel, which step of the funnel). Part 2 asks **why**.

# PART 2. Root-cause drill-down: why are booked tests short?

We test a tree of possible causes. For each branch we show what the data says, and what it **cannot** say.

**Booked tests = active patients x bookings per patient x tests per booking**

| Branch | Cause tested |
|---|---|
| R1 | Was the plan itself realistic? |
| R2 | Which lever fell short: patients, frequency or basket? |
| R3 | Patient flow: retained, lost, new, returning |
| R4 | How does each city compare with its peers? |
| R5 | Channel (clinic vs home) and campaigns |
| R6 | Individual clinics and capacity |
| R7 | Test mix |
| R8 | Did the new booking system (nova) cutover hurt? |
| R9 | Are nova patients re-keyed under new IDs? |
| R10 | Lead time (capacity pressure) |

Part 2 reuses the tables from Part 1 (`df_booked`, `df_compare`, `city_table`, ...), so **run Part 1 first**.

## 2.1 R1: Was the plan realistic?

- How much growth did the plan ask for in each city, and does it vary by month?
- Is the growth the plan asked for related to the growth that actually happened? (`.corr()` across cities. A low number means the plan ignored city reality)
- Did attainment **improve, fade or stay flat**? We compare the average attainment of the first 2 months with the last 2 months

A flat attainment from the first month means the target was **too high from the start**, not that performance decayed during the year.

In [137]:
prior_cm = df_booked[df_booked["period"] == "prior"].pivot_table(
    index="city", columns="compare_month", values="tests", aggfunc="sum", fill_value=0)

plan_growth_cm = (plan_cm / prior_cm - 1) * 100
plan_growth_cm.round(1)

month,2026-01,2026-02,2026-03,2026-04,2026-05,2026-06
city,,,,,,
Ahmedabad,18.00,18.00,18.00,18.00,18.00,18.00
Bengaluru,20.00,20.00,20.00,20.00,20.00,20.00
Chennai,17.00,17.00,17.00,17.00,17.00,17.00
Hyderabad,18.00,18.00,18.00,18.00,18.00,18.00
Jaipur,16.00,16.00,16.00,16.00,16.00,16.00
Pune,19.00,19.00,19.00,19.00,19.00,19.00


In [138]:
r1 = pd.DataFrame({
    "plan_growth_avg_pct": plan_growth_cm.mean(axis=1),
    "plan_growth_std_across_months": plan_growth_cm.std(axis=1),
    "actual_growth_pct": city_table.drop("TOTAL")["actual_growth_pct"],
    "attainment_first2_pct": attainment_cm[cur_months[:2]].mean(axis=1),
    "attainment_last2_pct": attainment_cm[cur_months[-2:]].mean(axis=1),
})
r1["attainment_change_pp"] = r1["attainment_last2_pct"] - r1["attainment_first2_pct"]

display(r1.round(1))
print("Correlation between plan growth and actual growth across cities:", round(r1["plan_growth_avg_pct"].corr(r1["actual_growth_pct"]), 2))

first2 = monthly_totals["attainment_pct"].iloc[:2].mean()
last2 = monthly_totals["attainment_pct"].iloc[-2:].mean()
print(f"Network attainment: first 2 months {first2:.1f}% -> last 2 months {last2:.1f}%")

,plan_growth_avg_pct,plan_growth_std_across_months,actual_growth_pct,attainment_first2_pct,attainment_last2_pct,attainment_change_pp
city,,,,,,
Ahmedabad,18.00,0.00,10.60,95.40,92.80,-2.60
Bengaluru,20.00,0.00,7.60,93.80,87.80,-6.00
Chennai,17.00,0.00,0.60,93.10,83.60,-9.50
Hyderabad,18.00,0.00,7.40,90.40,92.00,1.50
Jaipur,16.00,0.00,-10.90,74.50,80.70,6.20
Pune,19.00,0.00,8.10,90.50,87.80,-2.70


Correlation between plan growth and actual growth across cities: 0.79
Network attainment: first 2 months 90.9% -> last 2 months 87.9%


## 2.2 R2: Which lever fell short?

Booked tests = **patients** x **bookings per patient** (frequency) x **tests per booking** (basket).

We compute each lever for prior and current and its growth. Then we answer a practical question: **how much of each lever alone would have been needed to close the gap?**

Again we stack a `city = "TOTAL"` copy, so distinct patients are counted correctly in the total.

In [139]:
df_compare_total = df_compare.copy()
df_compare_total["city"] = "TOTAL"
df_compare_all = pd.concat([df_compare, df_compare_total])

patients_p = df_compare_all.pivot_table(index="city", columns="period", values="patient_id", aggfunc="nunique")
bookings_p = df_compare_all.pivot_table(index="city", columns="period", values="booking_id", aggfunc="count")
tests_p = df_compare_all.pivot_table(index="city", columns="period", values="tests", aggfunc="sum")

patients_p

period,current,prior
city,,
Ahmedabad,18326,16705
Bengaluru,23237,21696
Chennai,16689,17570
Hyderabad,16990,15717
Jaipur,8530,9711
Pune,14348,13183
TOTAL,98120,94582


In [140]:
lever_table = pd.DataFrame({
    "patients_prior": patients_p["prior"], "patients_current": patients_p["current"],
    "bookings_per_patient_prior": bookings_p["prior"] / patients_p["prior"],
    "bookings_per_patient_current": bookings_p["current"] / patients_p["current"],
    "tests_per_booking_prior": tests_p["prior"] / bookings_p["prior"],
    "tests_per_booking_current": tests_p["current"] / bookings_p["current"],
    "tests_prior": tests_p["prior"], "tests_current": tests_p["current"],
})

lever_table["patients_growth_pct"] = (lever_table["patients_current"] / lever_table["patients_prior"] - 1) * 100
lever_table["frequency_growth_pct"] = (lever_table["bookings_per_patient_current"] / lever_table["bookings_per_patient_prior"] - 1) * 100
lever_table["basket_growth_pct"] = (lever_table["tests_per_booking_current"] / lever_table["tests_per_booking_prior"] - 1) * 100
lever_table["tests_growth_pct"] = (lever_table["tests_current"] / lever_table["tests_prior"] - 1) * 100

lever_table[["patients_growth_pct", "frequency_growth_pct", "basket_growth_pct", "tests_growth_pct"]].round(1)

,patients_growth_pct,frequency_growth_pct,basket_growth_pct,tests_growth_pct
city,,,,
Ahmedabad,9.70,-1.00,1.90,10.60
Bengaluru,7.10,-2.30,2.80,7.60
Chennai,-5.00,3.10,2.70,0.60
Hyderabad,8.10,-2.50,1.90,7.40
Jaipur,-12.20,-0.30,1.70,-10.90
Pune,8.80,-2.30,1.60,8.10
TOTAL,3.70,-1.00,2.20,5.00


In [141]:
lever_table["plan_tests"] = city_table["current_plan"]
lever_table["extra_tests_needed"] = lever_table["plan_tests"] - lever_table["tests_current"]

# % increase needed to reach the plan, if only ONE lever moved
need = lever_table["plan_tests"] / lever_table["tests_current"] - 1
lever_table["extra_patients_needed_alone"] = lever_table["patients_current"] * need
lever_table["extra_bookings_per_patient_needed_alone"] = lever_table["bookings_per_patient_current"] * need
lever_table["extra_tests_per_booking_needed_alone"] = lever_table["tests_per_booking_current"] * need

lever_table[["plan_tests", "tests_current", "extra_tests_needed", "extra_patients_needed_alone",
             "extra_bookings_per_patient_needed_alone", "extra_tests_per_booking_needed_alone"]].round(2)

,plan_tests,tests_current,extra_tests_needed,extra_patients_needed_alone,extra_bookings_per_patient_needed_alone,extra_tests_per_booking_needed_alone
city,,,,,,
Ahmedabad,90773,85084,5689,"1,225.34",0.13,0.16
Bengaluru,120085,107661,12424,"2,681.53",0.23,0.27
Chennai,93912,80772,13140,"2,714.97",0.33,0.38
Hyderabad,86421,78674,7747,"1,673.00",0.19,0.23
Jaipur,52372,40236,12136,"2,572.82",0.61,0.71
Pune,72177,65575,6602,"1,444.54",0.20,0.24
TOTAL,515740,458002,57738,"12,369.49",0.25,0.30


## 2.3 R3: Patient flow (retained, lost, new, returning)

For each patient we count the tests in the prior and current windows, split into clinic and home. Then we label each patient:

| Group | Rule |
|---|---|
| retained | tests in both windows |
| lost | tests last year, none now |
| new | tests now only, and signed up inside the window |
| returning_inactive_prior | tests now only, but signed up earlier (inactive last year) |

If retained patients lose clinic tests and gain home tests, they are **switching channels**, not leaving.

New idea: first create four helper columns with `np.where`, then one `groupby` adds them up per patient.

In [142]:
is_prior = df_compare["period"] == "prior"
is_current = df_compare["period"] == "current"
is_clinic = df_compare["channel"] == "clinic"
is_home = df_compare["channel"] == "home"

df_compare["prior_clinic"] = np.where(is_prior & is_clinic, df_compare["tests"], 0)
df_compare["prior_home"] = np.where(is_prior & is_home, df_compare["tests"], 0)
df_compare["current_clinic"] = np.where(is_current & is_clinic, df_compare["tests"], 0)
df_compare["current_home"] = np.where(is_current & is_home, df_compare["tests"], 0)

df_compare[["patient_id", "period", "channel", "tests", "prior_clinic", "prior_home", "current_clinic", "current_home"]].head()

,patient_id,period,channel,tests,prior_clinic,prior_home,current_clinic,current_home
0,PAHM0028525,prior,clinic,3,3,0,0,0
1,PAHM0031384,prior,clinic,1,1,0,0,0
2,PAHM0027001,prior,home,4,0,4,0,0
3,PAHM0007069,prior,clinic,1,1,0,0,0
4,PAHM0002520,prior,clinic,2,2,0,0,0


In [143]:
df_patients = df_compare.groupby(["city", "patient_id"]).agg(
    prior_clinic=("prior_clinic", "sum"), prior_home=("prior_home", "sum"),
    current_clinic=("current_clinic", "sum"), current_home=("current_home", "sum")).reset_index()

df_patients["prior_tests"] = df_patients["prior_clinic"] + df_patients["prior_home"]
df_patients["current_tests"] = df_patients["current_clinic"] + df_patients["current_home"]

df_patients.head()

,city,patient_id,prior_clinic,prior_home,current_clinic,current_home,prior_tests,current_tests
0,Ahmedabad,PAHM0000009,0,0,1,2,0,3
1,Ahmedabad,PAHM0000013,0,6,0,3,6,3
2,Ahmedabad,PAHM0000016,5,0,0,0,5,0
3,Ahmedabad,PAHM0000023,1,4,0,0,5,0
4,Ahmedabad,PAHM0000027,4,0,0,0,4,0


Signup date of each patient. If the signup date is blank we use the first booking date instead.

In [144]:
df_patient_dates = df_booked.groupby(["city", "patient_id"]).agg(
    signup=("patient_signup_date", "min"), first_seen=("slot_at", "min")).reset_index()

df_patients = df_patients.merge(df_patient_dates, on=["city", "patient_id"], how="left")
df_patients["signup_final"] = df_patients["signup"].fillna(df_patients["first_seen"])

df_patients.head()

,city,patient_id,prior_clinic,prior_home,current_clinic,current_home,prior_tests,current_tests,signup,first_seen,signup_final
0,Ahmedabad,PAHM0000009,0,0,1,2,0,3,2023-12-26,2026-01-03 07:24:00,2023-12-26
1,Ahmedabad,PAHM0000013,0,6,0,3,6,3,2024-05-30,2025-01-07 12:32:00,2024-05-30
2,Ahmedabad,PAHM0000016,5,0,0,0,5,0,2024-11-23,2025-03-22 06:20:00,2024-11-23
3,Ahmedabad,PAHM0000023,1,4,0,0,5,0,2024-02-11,2025-02-06 11:54:00,2024-02-11
4,Ahmedabad,PAHM0000027,4,0,0,0,4,0,2024-09-14,2025-02-12 12:35:00,2024-09-14


In [145]:
window_start = pd.to_datetime(cur_months[0] + "-01")

has_current = df_patients["current_tests"] > 0
has_prior = df_patients["prior_tests"] > 0

df_patients["group"] = "other"
df_patients.loc[has_current & has_prior, "group"] = "retained"
df_patients.loc[~has_current & has_prior, "group"] = "lost"
df_patients.loc[has_current & ~has_prior, "group"] = "returning_inactive_prior"
df_patients.loc[has_current & ~has_prior & (df_patients["signup_final"] >= window_start), "group"] = "new"

df_patients["d_clinic"] = df_patients["current_clinic"] - df_patients["prior_clinic"]
df_patients["d_home"] = df_patients["current_home"] - df_patients["prior_home"]
df_patients["d_tests"] = df_patients["current_tests"] - df_patients["prior_tests"]

df_patients["group"].value_counts()

,count
group,
lost,63492
new,39287
retained,31090
returning_inactive_prior,27743


In [146]:
# stack a TOTAL copy again, so the total row is a real total
df_patients_total = df_patients.copy()
df_patients_total["city"] = "TOTAL"
df_patients_all = pd.concat([df_patients, df_patients_total])

by_group = df_patients_all.groupby(["city", "group"]).agg(
    patients=("patient_id", "count"), prior_tests=("prior_tests", "sum"), current_tests=("current_tests", "sum"),
    d_tests=("d_tests", "sum"), d_clinic_tests=("d_clinic", "sum"), d_home_tests=("d_home", "sum"))

by_group

patients  prior_tests  current_tests  d_tests  d_clinic_tests  d_home_tests
city      group                                                                                                
Ahmedabad lost                         11235        41448              0   -41448          -27038        -14410
          new                           7983            0          36320    36320           20757         15563
          retained                      5470        35478          21636   -13842          -11684         -2158
          returning_inactive_prior      4873            0          27128    27128           14988         12140
Bengaluru lost                         14637        54167              0   -54167          -35806        -18361
          new                          10068            0          45871    45871           25824         20047
          retained                      7059        45903          27674   -18229          -15252         -2977
          returning_inactive_prior      6110            0          34116    34116           18565         15551
Chennai   lost                         11602        42190              0   -42190          -27824        -14366
          new                           5131            0          24925    24925           14089         10836
          retained                      5968        38077          24378   -13699          -11880         -1819
          returning_inactive_prior      5590            0          31469    31469           17280         14189
Hyderabad lost                         10642        39887              0   -39887          -26077        -13810
          new                           7366            0          33282    33282           18828         14454
          retained                      5075        33350          20067   -13283          -11200         -2083
          returning_inactive_prior      4549            0          25325    25325           13648         11677
Jaipur    lost                          6521        24166              0   -24166          -15946         -8220
          new                           2575            0          12104    12104            6842          5262
          retained                      3190        20983          12912    -8071           -6584         -1487
          returning_inactive_prior      2765            0          15220    15220            8341          6879
Pune      lost                          8855        32568              0   -32568          -21428        -11140
          new                           6164            0          27657    27657           15404         12253
          retained                      4328        28085          16811   -11274           -9331         -1943
          returning_inactive_prior      3856            0          21107    21107           11419          9688
TOTAL     lost                         63492       234426              0  -234426         -154119        -80307
          new                          39287            0         180159   180159          101744         78415
          retained                     31090       201876         123478   -78398          -65931        -12467
          returning_inactive_prior     27743            0         154365   154365           84241         70124

**Rates per 100 prior-year active patients** (prior-year active = retained + lost):
- retention % = retained / prior active
- new per 100 = new / prior active x 100
- returning per 100 = returning / prior active x 100

In [147]:
group_counts = df_patients_all.pivot_table(index="city", columns="group", values="patient_id", aggfunc="count", fill_value=0)
group_counts = group_counts.reindex(columns=["retained", "lost", "new", "returning_inactive_prior"], fill_value=0)

rates = pd.DataFrame({
    "prior_active_patients": group_counts["retained"] + group_counts["lost"],
    "retained": group_counts["retained"],
    "lost": group_counts["lost"],
    "new": group_counts["new"],
    "returning": group_counts["returning_inactive_prior"],
})
rates["retention_rate_pct"] = rates["retained"] / rates["prior_active_patients"] * 100
rates["new_per_100_prior_active"] = rates["new"] / rates["prior_active_patients"] * 100
rates["returning_per_100_prior_active"] = rates["returning"] / rates["prior_active_patients"] * 100

rates.round(1)

,prior_active_patients,retained,lost,new,returning,retention_rate_pct,new_per_100_prior_active,returning_per_100_prior_active
city,,,,,,,,
Ahmedabad,16705,5470,11235,7983,4873,32.70,47.80,29.20
Bengaluru,21696,7059,14637,10068,6110,32.50,46.40,28.20
Chennai,17570,5968,11602,5131,5590,34.00,29.20,31.80
Hyderabad,15717,5075,10642,7366,4549,32.30,46.90,28.90
Jaipur,9711,3190,6521,2575,2765,32.80,26.50,28.50
Pune,13183,4328,8855,6164,3856,32.80,46.80,29.20
TOTAL,94582,31090,63492,39287,27743,32.90,41.50,29.30


### R3b: New patients by signup month, and year-on-year

A drop in new-patient signups points to an **acquisition** problem (as opposed to retention).

In [148]:
df_patients["signup_month"] = df_patients["signup_final"].dt.to_period("M").astype(str)
all_months = pd.period_range(prior_months[0], cur_months[-1], freq="M").astype(str)

new_counts = df_patients.pivot_table(index="city", columns="signup_month", values="patient_id", aggfunc="count", fill_value=0)
new_counts = new_counts.reindex(columns=all_months, fill_value=0)
new_counts.loc["TOTAL"] = new_counts.sum()

new_counts

,2025-01,2025-02,2025-03,2025-04,2025-05,2025-06,2025-07,2025-08,2025-09,2025-10,2025-11,2025-12,2026-01,2026-02,2026-03,2026-04,2026-05,2026-06
city,,,,,,,,,,,,,,,,,,
Ahmedabad,1123,1085,1091,1151,1108,1127,562,511,616,626,615,669,1313,1370,1347,1308,1320,1325
Bengaluru,1489,1446,1525,1412,1445,1427,692,717,767,794,735,808,1697,1835,1618,1667,1664,1587
Chennai,1146,1124,1223,1166,1166,1165,614,616,680,680,690,743,866,929,850,835,836,815
Hyderabad,1089,1091,1044,1086,1036,1005,518,506,556,562,584,628,1230,1282,1237,1242,1161,1214
Jaipur,667,699,721,589,639,607,304,325,334,357,378,368,437,439,441,416,415,427
Pune,903,895,855,889,873,853,438,438,505,472,500,519,1079,1014,1070,1038,944,1019
TOTAL,6417,6340,6459,6293,6267,6184,3128,3113,3458,3491,3502,3735,6622,6869,6563,6506,6340,6387


In [149]:
new_yoy = pd.DataFrame(index=new_counts.index)

for month in cur_months:
    last_year_month = str(pd.Period(month, "M") - 12)
    new_yoy[month] = (new_counts[month] / new_counts[last_year_month] - 1) * 100

new_yoy.replace([np.inf, -np.inf], np.nan).round(1)

,2026-01,2026-02,2026-03,2026-04,2026-05,2026-06
city,,,,,,
Ahmedabad,16.90,26.30,23.50,13.60,19.10,17.60
Bengaluru,14.00,26.90,6.10,18.10,15.20,11.20
Chennai,-24.40,-17.30,-30.50,-28.40,-28.30,-30.00
Hyderabad,12.90,17.50,18.50,14.40,12.10,20.80
Jaipur,-34.50,-37.20,-38.80,-29.40,-35.10,-29.70
Pune,19.50,13.30,25.10,16.80,8.10,19.50
TOTAL,3.20,8.30,1.60,3.40,1.20,3.30


## 2.4 R4: Each city vs its peers

For every metric we compute **city minus the median of the OTHER cities**. The median is not pulled by one extreme city, and leaving the city itself out stops it from influencing its own benchmark. The most negative lever shows what that city is worst at.

New idea: a small `for` loop over the cities.

In [150]:
peer_values = pd.DataFrame({
    "tests_growth_pct": lever_table["tests_growth_pct"],
    "patients_growth_pct": lever_table["patients_growth_pct"],
    "frequency_growth_pct": lever_table["frequency_growth_pct"],
    "basket_growth_pct": lever_table["basket_growth_pct"],
    "retention_rate_pct": rates["retention_rate_pct"],
    "new_per_100_prior_active": rates["new_per_100_prior_active"],
    "returning_per_100_prior_active": rates["returning_per_100_prior_active"],
}).drop("TOTAL")

peer_values.round(1)

,tests_growth_pct,patients_growth_pct,frequency_growth_pct,basket_growth_pct,retention_rate_pct,new_per_100_prior_active,returning_per_100_prior_active
city,,,,,,,
Ahmedabad,10.60,9.70,-1.00,1.90,32.70,47.80,29.20
Bengaluru,7.60,7.10,-2.30,2.80,32.50,46.40,28.20
Chennai,0.60,-5.00,3.10,2.70,34.00,29.20,31.80
Hyderabad,7.40,8.10,-2.50,1.90,32.30,46.90,28.90
Jaipur,-10.90,-12.20,-0.30,1.70,32.80,26.50,28.50
Pune,8.10,8.80,-2.30,1.60,32.80,46.80,29.20


In [151]:
peer_gap = peer_values.copy()

for city in peer_values.index:
    other_cities = peer_values.drop(city)
    peer_gap.loc[city] = peer_values.loc[city] - other_cities.median()

peer_gap.round(1)

,tests_growth_pct,patients_growth_pct,frequency_growth_pct,basket_growth_pct,retention_rate_pct,new_per_100_prior_active,returning_per_100_prior_active
city,,,,,,,
Ahmedabad,3.20,2.60,1.20,0.00,-0.10,1.40,0.20
Bengaluru,0.20,-1.00,-1.20,0.90,-0.30,-0.40,-1.00
Chennai,-7.00,-13.10,5.40,0.90,1.20,-17.60,2.90
Hyderabad,-0.20,1.00,-1.40,-0.00,-0.50,0.50,-0.20
Jaipur,-18.50,-20.30,2.00,-0.20,0.10,-20.20,-0.70
Pune,0.70,1.70,-1.20,-0.20,0.10,0.40,0.30


## 2.5 R5: Channel and campaigns

Compare clinic vs home year-on-year growth in the **first half** and **last half** of the window, to see whether a channel is fading or surging. Also: what share of current home tests came from **campaign** bookings (discount or waiver campaigns) in each half?

In [152]:
half = len(cur_months) // 2
df_compare["window"] = np.where(df_compare["compare_month"].isin(cur_months[:half]), "first_half", "last_half")

df_compare[["compare_month", "window"]].drop_duplicates().sort_values("compare_month")

,compare_month,window
0,2026-01,first_half
5616,2026-02,first_half
11044,2026-03,first_half
16500,2026-04,last_half
22258,2026-05,last_half
27799,2026-06,last_half


In [153]:
first_half = df_compare[df_compare["window"] == "first_half"].pivot_table(
    index=["city", "channel"], columns="period", values="tests", aggfunc="sum", fill_value=0)
last_half = df_compare[df_compare["window"] == "last_half"].pivot_table(
    index=["city", "channel"], columns="period", values="tests", aggfunc="sum", fill_value=0)

channel_yoy = pd.DataFrame({
    "yoy_first_half_pct": (first_half["current"] / first_half["prior"] - 1) * 100,
    "yoy_last_half_pct": (last_half["current"] / last_half["prior"] - 1) * 100,
})
channel_yoy["change_pp"] = channel_yoy["yoy_last_half_pct"] - channel_yoy["yoy_first_half_pct"]

channel_yoy.round(1)

yoy_first_half_pct  yoy_last_half_pct  change_pp
city      channel                                                  
Ahmedabad clinic               -12.40               0.50      13.00
          home                  63.00              22.10     -40.80
Bengaluru clinic               -16.90              -3.00      13.90
          home                  57.00              26.30     -30.70
Chennai   clinic               -18.90             -12.60       6.30
          home                  46.50              17.80     -28.70
Hyderabad clinic               -15.90              -3.90      11.90
          home                  52.50              28.60     -23.90
Jaipur    clinic               -33.90             -14.30      19.60
          home                  22.80               7.90     -15.00
Pune      clinic               -16.20              -3.30      12.90
          home                  62.70              22.70     -40.00

In [154]:
# the whole network, by channel
first_half_net = df_compare[df_compare["window"] == "first_half"].pivot_table(index="channel", columns="period", values="tests", aggfunc="sum")
last_half_net = df_compare[df_compare["window"] == "last_half"].pivot_table(index="channel", columns="period", values="tests", aggfunc="sum")

channel_yoy_net = pd.DataFrame({
    "yoy_first_half_pct": (first_half_net["current"] / first_half_net["prior"] - 1) * 100,
    "yoy_last_half_pct": (last_half_net["current"] / last_half_net["prior"] - 1) * 100,
})
channel_yoy_net["change_pp"] = channel_yoy_net["yoy_last_half_pct"] - channel_yoy_net["yoy_first_half_pct"]

channel_yoy_net.round(1)

,yoy_first_half_pct,yoy_last_half_pct,change_pp
channel,,,
clinic,-18.10,-5.50,12.60
home,52.40,22.00,-30.40


In [155]:
home_current = df_compare[(df_compare["period"] == "current") & (df_compare["channel"] == "home")].copy()
home_current["campaign_tests"] = np.where(home_current["campaign"], home_current["tests"], 0)

campaign_share = home_current.groupby("window").agg(home_tests=("tests", "sum"), campaign_tests=("campaign_tests", "sum"))
campaign_share["campaign_share_pct"] = campaign_share["campaign_tests"] / campaign_share["home_tests"] * 100

campaign_share.round(1)

,home_tests,campaign_tests,campaign_share_pct
window,,,
first_half,115028,115028,100.00
last_half,89853,0,0.00


## 2.6 R6: Clinics

Per clinic:
- tests prior vs current, and growth
- **capacity use** = bookings on a day / slots per day (mean, and the 95th percentile = a busy day). If a clinic's busy days sit at or above 90%, it is **capacity-bound**
- first and last booking date (is it a new clinic, or did it stop?)
- each declining clinic's **share of its city's decline**

If clinics are full they cannot grow, however demand behaves. That is a very different problem from weak demand.

In [156]:
clinic_table = df_compare.pivot_table(index=["city", "clinic_id"], columns="period", values="tests", aggfunc="sum", fill_value=0)
clinic_table = clinic_table.reset_index()
clinic_table.columns.name = None

clinic_table["yoy_pct"] = (clinic_table["current"] / clinic_table["prior"].replace(0, np.nan) - 1) * 100
clinic_table["d_tests"] = clinic_table["current"] - clinic_table["prior"]

# only the declines (growth counts as 0), then each clinic's share of its city's total decline
clinic_table["decline"] = np.where(clinic_table["d_tests"] < 0, clinic_table["d_tests"], 0)
clinic_table["city_decline"] = clinic_table.groupby("city")["decline"].transform("sum")
clinic_table["share_of_city_decline_pct"] = clinic_table["decline"] / clinic_table["city_decline"].replace(0, np.nan) * 100

clinic_table.head()

,city,clinic_id,current,prior,yoy_pct,d_tests,decline,city_decline,share_of_city_decline_pct
0,Ahmedabad,AHM-01,29405,26097,12.68,3308,0,0,NaN
1,Ahmedabad,AHM-02,19592,17252,13.56,2340,0,0,NaN
2,Ahmedabad,AHM-03,15408,14099,9.28,1309,0,0,NaN
3,Ahmedabad,AHM-04,11360,10513,8.06,847,0,0,NaN
4,Ahmedabad,AHM-05,9319,8965,3.95,354,0,0,NaN


In [157]:
# capacity: bookings per day in the clinic channel, divided by the clinic's slots per day
clinic_visits = df_compare[df_compare["channel"] == "clinic"].copy()
clinic_visits["day"] = clinic_visits["slot_at"].dt.normalize()

daily = clinic_visits.groupby(["clinic_id", "period", "day"]).size().reset_index(name="bookings")
slots = df_booked.groupby("clinic_id").agg(slots_per_day=("clinic_slots_per_day", "first")).reset_index()

daily = daily.merge(slots, on="clinic_id", how="left")
daily["utilisation"] = daily["bookings"] / daily["slots_per_day"]

daily.head()

,clinic_id,period,day,bookings,slots_per_day,utilisation
0,AHM-01,current,2026-01-01,25,60,0.42
1,AHM-01,current,2026-01-02,32,60,0.53
2,AHM-01,current,2026-01-03,37,60,0.62
3,AHM-01,current,2026-01-04,41,60,0.68
4,AHM-01,current,2026-01-05,43,60,0.72


In [158]:
# mean and 95th percentile of the daily utilisation, separately for each period
util_mean = daily.groupby(["clinic_id", "period"])["utilisation"].mean().reset_index(name="util_mean")
util_p95 = daily.groupby(["clinic_id", "period"])["utilisation"].quantile(0.95).reset_index(name="util_p95")
util = util_mean.merge(util_p95, on=["clinic_id", "period"])

util.head()

,clinic_id,period,util_mean,util_p95
0,AHM-01,current,0.64,0.85
1,AHM-01,prior,0.69,0.87
2,AHM-02,current,0.65,0.88
3,AHM-02,prior,0.69,0.90
4,AHM-03,current,0.51,0.70


In [159]:
# one row per clinic, with the prior and the current numbers side by side
util_prior = util[util["period"] == "prior"].drop(columns="period").rename(
    columns={"util_mean": "util_mean_prior", "util_p95": "util_p95_prior"})
util_current = util[util["period"] == "current"].drop(columns="period").rename(
    columns={"util_mean": "util_mean_current", "util_p95": "util_p95_current"})

clinic_table = clinic_table.merge(util_prior, on="clinic_id", how="left")
clinic_table = clinic_table.merge(util_current, on="clinic_id", how="left")

# capacity-bound = busy days (95th percentile) at or above 90% of the slots, in either period
clinic_table["capacity_bound"] = clinic_table[["util_p95_prior", "util_p95_current"]].max(axis=1) >= 0.9

clinic_table[["city", "clinic_id", "util_mean_prior", "util_mean_current", "util_p95_prior", "util_p95_current", "capacity_bound"]].head()

,city,clinic_id,util_mean_prior,util_mean_current,util_p95_prior,util_p95_current,capacity_bound
0,Ahmedabad,AHM-01,0.69,0.64,0.87,0.85,False
1,Ahmedabad,AHM-02,0.69,0.65,0.90,0.88,True
2,Ahmedabad,AHM-03,0.55,0.51,0.72,0.70,False
3,Ahmedabad,AHM-04,0.41,0.37,0.57,0.55,False
4,Ahmedabad,AHM-05,0.36,0.31,0.53,0.45,False


In [160]:
# first and last booking per clinic (all months): was a clinic opened or closed inside the window?
clinic_dates = df_booked.groupby("clinic_id").agg(first_booking=("slot_at", "min"), last_booking=("slot_at", "max")).reset_index()
clinic_table = clinic_table.merge(clinic_dates, on="clinic_id", how="left")

window_end = pd.to_datetime(cur_months[-1] + "-01") + pd.offsets.MonthEnd(0)
clinic_table["new_in_window"] = clinic_table["first_booking"] >= window_start
clinic_table["stopped"] = clinic_table["last_booking"] < (window_end - pd.Timedelta(days=14))
clinic_table["is_declining"] = clinic_table["d_tests"] < 0

clinic_table

,city,clinic_id,current,prior,yoy_pct,d_tests,decline,city_decline,share_of_city_decline_pct,util_mean_prior,util_p95_prior,util_mean_current,util_p95_current,capacity_bound,first_booking,last_booking,new_in_window,stopped,is_declining
0,Ahmedabad,AHM-01,29405,26097,12.68,3308,0,0,NaN,0.69,0.87,0.64,0.85,False,2025-01-01 06:19:00,2026-06-30 19:56:00,False,False,False
1,Ahmedabad,AHM-02,19592,17252,13.56,2340,0,0,NaN,0.69,0.90,0.65,0.88,True,2025-01-01 06:29:00,2026-06-30 18:39:00,False,False,False
2,Ahmedabad,AHM-03,15408,14099,9.28,1309,0,0,NaN,0.55,0.72,0.51,0.70,False,2025-01-01 06:06:00,2026-06-30 19:57:00,False,False,False
3,Ahmedabad,AHM-04,11360,10513,8.06,847,0,0,NaN,0.41,0.57,0.37,0.55,False,2025-01-01 06:00:00,2026-06-30 18:33:00,False,False,False
4,Ahmedabad,AHM-05,9319,8965,3.95,354,0,0,NaN,0.36,0.53,0.31,0.45,False,2025-01-01 06:03:00,2026-06-30 19:11:00,False,False,False
5,Bengaluru,BLR-01,33136,31118,6.48,2018,0,0,NaN,0.83,1.02,0.72,0.92,True,2025-01-01 06:23:00,2026-06-30 19:59:00,False,False,False
6,Bengaluru,BLR-02,22742,20817,9.25,1925,0,0,NaN,0.84,1.10,0.73,0.97,True,2025-01-01 06:00:00,2026-06-30 19:41:00,False,False,False
7,Bengaluru,BLR-03,18138,16766,8.18,1372,0,0,NaN,0.66,0.85,0.59,0.80,False,2025-01-01 06:55:00,2026-06-30 19:54:00,False,False,False
8,Bengaluru,BLR-04,13092,12372,5.82,720,0,0,NaN,0.50,0.68,0.42,0.62,False,2025-01-01 06:03:00,2026-06-30 19:54:00,False,False,False
9,Bengaluru,BLR-05,11250,10514,7.00,736,0,0,NaN,0.42,0.60,0.36,0.53,False,2025-01-01 06:21:00,2026-06-30 19:34:00,False,False,False


In [161]:
# the 3 clinics with the biggest share of each city's decline
top3 = clinic_table.sort_values("share_of_city_decline_pct", ascending=False).groupby("city").head(3)
top3_share = top3.groupby("city").agg(top3_share_of_city_decline_pct=("share_of_city_decline_pct", "sum"))

clinic_summary = clinic_table.groupby("city").agg(
    clinics=("clinic_id", "count"), clinics_declining=("is_declining", "sum"),
    clinics_capacity_bound=("capacity_bound", "sum"), new_clinics=("new_in_window", "sum"),
    stopped_clinics=("stopped", "sum")).join(top3_share)

clinic_summary.round(1)

,clinics,clinics_declining,clinics_capacity_bound,new_clinics,stopped_clinics,top3_share_of_city_decline_pct
city,,,,,,
Ahmedabad,5,0,1,0,0,0.00
Bengaluru,6,0,2,0,0,0.00
Chennai,5,3,2,0,0,100.00
Hyderabad,5,0,1,0,0,0.00
Jaipur,5,4,0,1,0,85.10
Pune,5,0,0,0,0,0.00


## 2.7 R7: Test mix

For each `test_code`: prior, current, year-on-year growth, and the gap against **uniform plan growth** (what it would have been if every test had grown at the plan's overall rate). This is an **assumption** (it is in the column name), because the plan has no per-test targets. It shows which tests carry most of the gap.

In [162]:
# keep only these 2 columns of df_lines, so a "city" column in the lines file cannot clash with ours (city_x / city_y)
df_lines_compare = df_lines[["booking_id", "test_code"]].merge(
    df_compare[["booking_id", "city", "period"]], on="booking_id", how="inner")
df_lines_compare["test_code"] = df_lines_compare["test_code"].fillna("MISSING")

In [163]:
plan_growth = city_table.loc["TOTAL", "current_plan"] / city_table.loc["TOTAL", "prior_booked"] - 1
print(f"The plan asked for {plan_growth:.1%} growth overall")

mix = df_lines_compare.pivot_table(index="test_code", columns="period", values="booking_id", aggfunc="count", fill_value=0)
mix["yoy_pct"] = (mix["current"] / mix["prior"].replace(0, np.nan) - 1) * 100
mix["gap_vs_uniform_plan_growth_ASSUMPTION"] = mix["prior"] * (1 + plan_growth) - mix["current"]
mix["share_of_total_gap_pct"] = mix["gap_vs_uniform_plan_growth_ASSUMPTION"] / mix["gap_vs_uniform_plan_growth_ASSUMPTION"].sum() * 100

mix.sort_values("gap_vs_uniform_plan_growth_ASSUMPTION", ascending=False).round(1)

The plan asked for 18.2% growth overall


period,current,prior,yoy_pct,gap_vs_uniform_plan_growth_ASSUMPTION,share_of_total_gap_pct
test_code,,,,,
T001,54797,54359,0.80,"9,459.20",16.40
T002,44360,43603,1.70,"7,181.80",12.40
T004,32648,32565,0.30,"5,846.10",10.10
T007,29597,29656,-0.20,"5,458.50",9.50
T003,29820,29638,0.60,"5,214.20",9.00
T010,26361,26448,-0.30,"4,902.40",8.50
T005,23323,23249,0.30,"4,159.00",7.20
T008,20320,20074,1.20,"3,408.90",5.90
T006,20397,20126,1.30,"3,393.40",5.90


In [164]:
# the same growth, by city (cities down, test codes across)
mix_prior = df_lines_compare[df_lines_compare["period"] == "prior"].pivot_table(
    index="city", columns="test_code", values="booking_id", aggfunc="count", fill_value=0)
mix_current = df_lines_compare[df_lines_compare["period"] == "current"].pivot_table(
    index="city", columns="test_code", values="booking_id", aggfunc="count", fill_value=0)

mix_city_yoy = (mix_current / mix_prior.replace(0, np.nan) - 1) * 100
mix_city_yoy.round(1)

test_code,MISSING,PK01,PK02,PK03,PK04,T001,T002,T003,T004,T005,T006,T007,T008,T009,T010,T011,T012,T013,T014,T015,T016,T017,T018,T019,T020
city,,,,,,,,,,,,,,,,,,,,,,,,,
Ahmedabad,34.20,43.30,39.00,41.80,39.70,4.20,11.40,5.70,3.40,9.30,3.50,3.40,9.70,5.80,5.70,7.40,4.50,5.30,6.30,1.70,8.40,8.70,2.80,6.10,8.00
Bengaluru,49.40,36.20,36.90,34.00,32.50,4.40,2.30,3.00,4.50,2.70,1.10,3.40,2.10,3.10,4.60,7.60,1.80,4.60,6.00,1.40,5.00,2.70,1.20,10.10,6.60
Chennai,-25.00,30.60,27.20,24.60,35.10,-4.00,-5.00,-0.80,-2.80,-8.60,-2.50,-2.80,0.00,-2.10,-3.60,-3.20,3.10,-6.30,-1.40,-7.50,-0.90,-4.40,1.70,3.20,-5.80
Hyderabad,32.00,37.70,43.30,36.80,42.30,2.80,5.80,2.80,1.60,6.00,5.30,2.30,4.40,5.40,-0.10,2.20,-3.70,2.90,-1.20,-2.00,0.10,6.80,1.40,2.30,3.80
Jaipur,-59.30,16.90,17.40,20.80,14.80,-12.70,-12.10,-17.60,-16.10,-16.30,-13.50,-15.80,-15.90,-11.10,-16.00,-9.60,-21.70,-16.10,-16.10,-22.60,-16.60,-12.10,-10.60,-6.30,-16.80
Pune,4.30,45.90,35.30,42.70,29.50,4.50,2.90,3.40,3.70,2.50,10.50,1.20,0.00,7.00,0.10,2.40,10.80,4.10,7.10,4.80,1.40,3.50,-3.20,-1.90,17.90


## 2.8 R8: Did the nova cutover hurt? (difference-in-differences)

The new booking system ("nova") went live in some cities at some point. We compare:

1. **Nova cities** vs **other cities**: year-on-year growth by month
2. Each group's growth **before** vs **after** go-live
3. The **difference in differences** = (nova change) minus (other cities' change)

The second subtraction removes whatever happened to everyone (seasonality, market). What remains is the extra change in the nova cities. It is **descriptive**: it shows a pattern consistent with a cutover effect, it does not prove one.

In [165]:
is_nova = df_booked["source_system"] == "nova"
nova_cities = sorted(df_booked.loc[is_nova, "city"].unique())
first_nova_month = df_booked.loc[is_nova, "slot_at"].min().strftime("%Y-%m")

pre_months = [m for m in cur_months if m < first_nova_month]
post_months = [m for m in cur_months if m >= first_nova_month]

print("Nova cities:", nova_cities)
print("First nova booking month:", first_nova_month)
print("Months before go-live:", pre_months)
print("Months after go-live: ", post_months)

Nova cities: ['Chennai', 'Jaipur']
First nova booking month: 2026-04
Months before go-live: ['2026-01', '2026-02', '2026-03']
Months after go-live:  ['2026-04', '2026-05', '2026-06']


In [166]:
df_compare["city_group"] = np.where(df_compare["city"].isin(nova_cities), "nova_cities", "other_cities")

monthly_group = df_compare.pivot_table(index=["city_group", "compare_month"], columns="period", values="tests", aggfunc="sum").reset_index()
monthly_group["yoy_pct"] = (monthly_group["current"] / monthly_group["prior"] - 1) * 100

monthly_yoy = monthly_group.pivot(index="compare_month", columns="city_group", values="yoy_pct")
monthly_yoy["gap_pp"] = monthly_yoy["nova_cities"] - monthly_yoy["other_cities"]

monthly_yoy.round(1)

city_group,nova_cities,other_cities,gap_pp
compare_month,,,
2026-01,-1.90,7.80,-9.80
2026-02,2.70,12.60,-9.90
2026-03,-9.70,8.80,-18.50
2026-04,-4.40,7.20,-11.50
2026-05,-4.40,5.80,-10.20
2026-06,-3.10,8.10,-11.10


In [167]:
if len(pre_months) == 0 or len(post_months) == 0:
    print("Nova was live for the whole window (or not at all), so a before/after comparison is not possible")
else:
    df_compare["go_live"] = np.where(df_compare["compare_month"].isin(post_months), "post", "pre")

    phase_group = df_compare.pivot_table(index=["city_group", "go_live"], columns="period", values="tests", aggfunc="sum").reset_index()
    phase_group["yoy_pct"] = (phase_group["current"] / phase_group["prior"] - 1) * 100

    did = phase_group.pivot(index="city_group", columns="go_live", values="yoy_pct")
    did["change_pp"] = did["post"] - did["pre"]
    display(did.round(1))

    did_value = did.loc["nova_cities", "change_pp"] - did.loc["other_cities", "change_pp"]
    print(f"Difference-in-differences: {did_value:+.1f} pp (nova cities' change minus other cities' change)")

go_live,post,pre,change_pp
city_group,,,
nova_cities,-3.90,-3.10,-0.80
other_cities,7.00,9.70,-2.70


Difference-in-differences: +1.9 pp (nova cities' change minus other cities' change)


## 2.9 R9: Are nova patients re-keyed under new IDs?

If nova gave **new patient IDs to existing patients**, then existing patients would look like "new" and "lost" patients in R3, and R3 would mislead us. By nova city we check:
- how many nova patient IDs are **not found** in the legacy system
- how many of those claim a signup date **before** go-live (a re-keyed patient would)
- how many of those share a **phone number** with a legacy patient (only counts are shown, no phone numbers)

In [168]:
# keep only the last 10 digits of the phone number, so "+91 98765-43210" and "9876543210" match
df_booked["phone_clean"] = df_booked["patient_phone"].astype(str).str.replace(r"\D", "", regex=True).str[-10:]
df_booked["phone_clean"] = df_booked["phone_clean"].replace("", np.nan)

df_nova = df_booked[df_booked["source_system"] == "nova"]
df_legacy = df_booked[df_booked["source_system"] == "legacy"]

legacy_ids = set(df_legacy["patient_id"])
phone_of_patient = df_booked.dropna(subset=["phone_clean"]).groupby("patient_id")["phone_clean"].first()
legacy_phones = set(phone_of_patient[phone_of_patient.index.isin(legacy_ids)])

print(len(df_nova), "nova bookings;", len(legacy_ids), "legacy patient ids;", len(legacy_phones), "legacy phone numbers")

25043 nova bookings; 183878 legacy patient ids; 183869 legacy phone numbers


In [169]:
results = []

for city in nova_cities:
    d = df_nova[df_nova["city"] == city]
    go_live = d["slot_at"].min()

    nova_patients = d.groupby("patient_id").agg(signup=("patient_signup_date", "min")).reset_index()
    found_in_legacy = nova_patients["patient_id"].isin(legacy_ids)
    not_in_legacy = nova_patients[~found_in_legacy]

    phone_match = phone_of_patient.reindex(not_in_legacy["patient_id"]).isin(legacy_phones).sum()

    results.append({
        "city": city,
        "nova_patients": len(nova_patients),
        "share_id_found_in_legacy_pct": found_in_legacy.mean() * 100,
        "ids_not_in_legacy": len(not_in_legacy),
        "...of which signup before go-live": int((not_in_legacy["signup"] < go_live).sum()),
        "...of which phone found in legacy": int(phone_match),
    })

df_identity = pd.DataFrame(results).set_index("city")
df_identity.round(1)

,nova_patients,share_id_found_in_legacy_pct,ids_not_in_legacy,...of which signup before go-live,...of which phone found in legacy
city,,,,,
Chennai,10486,71.50,2984,520,0
Jaipur,5398,72.20,1498,251,0


## 2.10 R10: Lead time

Median days between booking and service date, by city and month. If the lead time **gets longer**, slots are getting scarce (capacity pressure). If it is stable, capacity is probably not the cause.

In [170]:
lead_time = df_booked.pivot_table(index="city", columns="month", values="lead_days", aggfunc="median")
lead_time.round(2)

month,2025-01,2025-02,2025-03,2025-04,2025-05,2025-06,2025-07,2025-08,2025-09,2025-10,2025-11,2025-12,2026-01,2026-02,2026-03,2026-04,2026-05,2026-06,2026-07
city,,,,,,,,,,,,,,,,,,,
Ahmedabad,2.41,3.01,2.40,2.41,2.39,2.40,2.41,2.40,3.00,2.40,3.01,3.02,3.01,3.00,3.01,2.40,2.39,3.01,NaN
Bengaluru,2.39,3.02,3.01,3.00,2.41,2.40,2.40,2.41,2.41,2.39,3.00,3.01,2.41,2.41,2.42,2.41,3.02,2.40,NaN
Chennai,2.41,3.01,2.40,3.03,2.41,3.01,2.41,3.00,3.02,3.02,2.41,2.41,3.00,3.01,2.42,3.00,3.08,3.04,5.10
Hyderabad,2.41,3.03,2.41,2.40,2.38,2.41,2.41,2.40,2.39,3.00,3.03,3.01,2.39,2.41,3.01,3.00,2.41,2.41,NaN
Jaipur,3.02,2.40,2.39,2.41,3.01,3.03,3.01,2.41,2.39,3.01,3.02,2.42,3.01,2.41,2.40,3.01,3.03,3.08,5.76
Pune,3.00,3.03,2.40,2.41,3.01,3.01,2.41,3.01,2.40,3.01,3.01,3.02,3.00,3.01,3.03,3.00,2.71,2.38,NaN


## 2.11 R0: Scorecard, computed facts (not conclusions)

One short printed line per hypothesis, plus a **LIMITS** line for what the data cannot test. Each line is a **hypothesis to weigh, not a finding**.

In [171]:
cities = city_table.drop("TOTAL").index.tolist()
total = lever_table.loc["TOTAL"]

print("[H1 plan realism] plan growth by city runs from "
      f"{r1['plan_growth_avg_pct'].min():.1f}% to {r1['plan_growth_avg_pct'].max():.1f}%; actual growth from "
      f"{r1['actual_growth_pct'].min():.1f}% to {r1['actual_growth_pct'].max():.1f}%; "
      f"correlation across cities {r1['plan_growth_avg_pct'].corr(r1['actual_growth_pct']):.2f}")
print(f"[H1 plan realism] network attainment: first 2 months {first2:.1f}% -> last 2 months {last2:.1f}%")

print(f"[H2 levers] network growth: patients {total['patients_growth_pct']:+.1f}%, frequency {total['frequency_growth_pct']:+.1f}%, "
      f"basket {total['basket_growth_pct']:+.1f}%; gap to plan {total['extra_tests_needed']:,.0f} tests = "
      f"{total['extra_patients_needed_alone']:,.0f} extra active patients if patients alone had to close it "
      f"(vs {rates.loc['TOTAL', 'new']:,.0f} new patients acquired)")

[H1 plan realism] plan growth by city runs from 16.0% to 20.0%; actual growth from -10.9% to 10.6%; correlation across cities 0.79
[H1 plan realism] network attainment: first 2 months 90.9% -> last 2 months 87.9%
[H2 levers] network growth: patients +3.7%, frequency -1.0%, basket +2.2%; gap to plan 57,738 tests = 12,369 extra active patients if patients alone had to close it (vs 39,287 new patients acquired)


In [172]:
for city in cities:
    lever_gaps = peer_gap.loc[city, ["patients_growth_pct", "frequency_growth_pct", "basket_growth_pct"]]
    print(f"[H3 peer gap] {city}: tests growth {peer_gap.loc[city, 'tests_growth_pct']:+.1f} pp vs peers; weakest lever "
          f"{lever_gaps.idxmin()} ({lever_gaps.min():+.1f}); retention {peer_gap.loc[city, 'retention_rate_pct']:+.1f} pp, "
          f"new patients {peer_gap.loc[city, 'new_per_100_prior_active']:+.1f} per 100")

[H3 peer gap] Ahmedabad: tests growth +3.2 pp vs peers; weakest lever basket_growth_pct (+0.0); retention -0.1 pp, new patients +1.4 per 100
[H3 peer gap] Bengaluru: tests growth +0.2 pp vs peers; weakest lever frequency_growth_pct (-1.2); retention -0.3 pp, new patients -0.4 per 100
[H3 peer gap] Chennai: tests growth -7.0 pp vs peers; weakest lever patients_growth_pct (-13.1); retention +1.2 pp, new patients -17.6 per 100
[H3 peer gap] Hyderabad: tests growth -0.2 pp vs peers; weakest lever frequency_growth_pct (-1.4); retention -0.5 pp, new patients +0.5 per 100
[H3 peer gap] Jaipur: tests growth -18.5 pp vs peers; weakest lever patients_growth_pct (-20.3); retention +0.1 pp, new patients -20.2 per 100
[H3 peer gap] Pune: tests growth +0.7 pp vs peers; weakest lever frequency_growth_pct (-1.2); retention +0.1 pp, new patients +0.4 per 100


In [173]:
retained_total = by_group.loc[("TOTAL", "retained")]
print(f"[H4 channel] retained patients: clinic tests {retained_total['d_clinic_tests']:+,.0f}, home tests {retained_total['d_home_tests']:+,.0f} "
      "(opposite signs suggest switching between channels)")
print("[H4 channel] campaign share of current home tests by window:", campaign_share["campaign_share_pct"].round(1).to_dict())

print(f"[H5 capacity] clinics capacity-bound: {int(clinic_summary['clinics_capacity_bound'].sum())} of {int(clinic_summary['clinics'].sum())}; "
      f"declining {int(clinic_summary['clinics_declining'].sum())}; new {int(clinic_summary['new_clinics'].sum())}, "
      f"stopped {int(clinic_summary['stopped_clinics'].sum())}")

top_tests = mix.sort_values("gap_vs_uniform_plan_growth_ASSUMPTION", ascending=False).head(3)
for test_code, row in top_tests.iterrows():
    print(f"[H6 test mix] {test_code}: {row['share_of_total_gap_pct']:.0f}% of the gap, YoY {row['yoy_pct']:+.1f}%")

[H4 channel] retained patients: clinic tests -65,931, home tests -12,467 (opposite signs suggest switching between channels)
[H4 channel] campaign share of current home tests by window: {'first_half': 100.0, 'last_half': 0.0}
[H5 capacity] clinics capacity-bound: 6 of 31; declining 7; new 1, stopped 0
[H6 test mix] T001: 16% of the gap, YoY +0.8%
[H6 test mix] T002: 12% of the gap, YoY +1.7%
[H6 test mix] T004: 10% of the gap, YoY +0.3%


In [174]:
if len(pre_months) > 0 and len(post_months) > 0:
    print(f"[H7 cutover] nova cities' YoY change after go-live minus other cities' = {did_value:+.1f} pp (descriptive)")

for city, row in df_identity.iterrows():
    print(f"[H8 identity] {city}: {int(row['ids_not_in_legacy']):,} nova patients not found in legacy, "
          f"{int(row['...of which signup before go-live']):,} of them claim a signup before go-live")

print("[LIMITS] the data covers 2025-01 to 2026-06 only, so retention and acquisition can be shown to DIFFER from peers, "
      "not to have WORSENED. There is no price, competitor, marketing-spend, staffing or phlebotomist-capacity data, "
      "so those causes cannot be tested here.")

[H7 cutover] nova cities' YoY change after go-live minus other cities' = +1.9 pp (descriptive)
[H8 identity] Chennai: 2,984 nova patients not found in legacy, 520 of them claim a signup before go-live
[H8 identity] Jaipur: 1,498 nova patients not found in legacy, 251 of them claim a signup before go-live
[LIMITS] the data covers 2025-01 to 2026-06 only, so retention and acquisition can be shown to DIFFER from peers, not to have WORSENED. There is no price, competitor, marketing-spend, staffing or phlebotomist-capacity data, so those causes cannot be tested here.


### How to read the scorecard

| Hypothesis | What it looks like in the numbers |
|---|---|
| Plan set too high | Growth is the same across cities and months and unrelated to actual growth by city (low correlation). Attainment is flat from month 1, so the gap was there from the start |
| Patient base | A city's patients lever and its retention / new rates sit well below its peers (R4): an acquisition or retention problem |
| Frequency or basket | The gap is in bookings per patient or tests per booking, not in patients |
| Channel shift | Retained patients lose clinic tests and gain home tests; home growth concentrated in campaign bookings |
| Capacity | Clinics at or near 90% busy-day use cannot grow; lead times getting longer (R10) |
| Product mix | One or two test codes carry most of the gap (R7) |
| Cutover | Nova cities' YoY gets worse after go-live compared with the others (R8). Always check R9 before trusting R3 |
| Re-keyed patients | Re-keyed nova patients would inflate "new" and "lost" in the nova cities |

## Recap: what we used

| Tool | Where |
|---|---|
| `read_csv`, `drop_duplicates`, `str.strip`, `to_datetime`, `to_numeric(errors="coerce")` | loading and cleaning every table |
| `groupby().size()`, `merge`, `fillna` | tests per booking, status lookups |
| `np.where` | period labels, funnel outcome, patient groups, helper columns |
| `pivot_table` | every "rows x columns" table |
| `.loc["TOTAL"] = .sum()`, or a stacked `TOTAL` copy | tables with a total row |
| `.join`, `.map`, `.transform("sum")` | adding city values, shares of a city total |
| `idxmin`, `rank`, `sort_values` | rankings |
| `for` loop over cities | peer gap (R4), identity (R9) |

